<a href="https://colab.research.google.com/github/peace0191/FateCode_UI_Full/blob/main/Signiel_4910E_Colab_Project_ipynb%EC%9D%98_%EC%82%AC%EB%B3%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Signiel_4910E_Colab_Project

완전히 정리해서 한 번에 쓰실 수 있게 다 묶어놨습니다 😊

아래 두 가지 파일만 받으시면 됩니다.


---



1. 전체 프로젝트 압축 파일 (이미지 + 코드 + PPT)

📦 Signiel_4910E_ALL.zip
→ 4910호 실사진 + E타입 관련 사진들 + Colab/파이참/VS용 코드 + 25페이지 PPT 뼈대 전부 포함

Download Signiel_4910E_ALL.zip

***압축 안에는 대략 이런 구조가 들어 있습니다:

# 1. 전체 프로젝트 압축 파일 (이미지 + 코드 + PPT)

Signiel_4910E_Colab_Project/


---


  ├─ README.txt
  ├─ Signiel_4910E_AI_Report_25pages.pptx   ← 25페이지 PPT 골격



---


  ├─ signiel_4910E_prototype_colab.py       ← Colab/파이참/VS 공용 코드


---


  └─ data/
      └─ raw_images/   ← 지금까지 주신 4910호 + E타입 JPG 이미지 모음


Colab에서 바로 쓰실 때

이 ZIP을 PC에 다운로드

구글 드라이브에 업로드 후 압축 해제

Colab에서 File > Open from Drive 로 폴더 연결

signiel_4910E_prototype_colab.py 열어서 셀 단위로 실행 (또는 새 노트북에 복붙)

# 2. Colab에서 사용할 핵심 코드 (텍스트 버전)

압축 안에 이미 signiel_4910E_prototype_colab.py 로 들어있지만,
바로 확인하시라고 여기에도 그대로 붙여드립니다.


---
*   4910E 참고 실사진(refs_4910E) → 프로토타입 벡터 생성
*   네이버 확정매물 / 후보 사진(candidates_4910E) → 유사도 계산
*   홍보/책자 이미지(stock_4910E) → “스톡/허위 가능성(fake_score)” 계산
*  premium_views / kitchen / bath / option 폴더 → 프리미엄 점수 계산
*   premium_views / kitchen / bath / option 폴더 → 프리미엄 점수 계산
---
*  📌 폴더는 직접 만드셔야 합니다 (이미지는 사용자가 분류해서 넣기)

*  data/refs_4910E → 4910호 실내 ‘진짜’ 사진
*   data/stock_4910E → 책자/홍보/스톡컷
*   data/candidates_4910E → 네이버 확정매물/후보 매물 사진
*   data/premium_views_4910E → 한강뷰/도심뷰 강한 컷
*   data/premium_kitchen_4910E → 주방 프리미엄 컷
*   data/premium_bath_4910E → 욕실/파우더룸 프리미엄 컷
*   data/premium_option_4910E → 옵션(붙박이장, 마감재 등) 컷

In [ ]:
"""
Signiel Residence 4910E – Image Prototype + Premium Scoring Demo
-----------------------------------------------------------------
이 스크립트는 다음을 수행합니다.

1) 4910호 E타입 실사진(refs_4910E)을 기반으로 "프로토타입 벡터" 생성
2) 네이버 확정매물/후보 사진(candidates_4910E)과의 유사도(코사인) 계산
3) 책자/홍보컷(stock_4910E)과의 유사도로 "허위·가짜 가능성" 점수 계산
4) 프리미엄 요소 뷰/주방/욕실/옵션 폴더(premium_*_4910E)를 이용한 프리미엄 점수 산출
5) Top 10 유사 매물에 대한 표/그래프 + PDF 리포트 생성

Colab 사용법:
- 런타임 유형: GPU (선택 사항, CPU도 동작은 가능)
- 왼쪽 파일 탭에 이 프로젝트 폴더를 업로드 후, 이 .py 파일을 열어
  셀 단위로 실행하거나, 복사해서 새 노트북에 붙여넣어 사용하세요.
"""

import os
import glob
from typing import List, Dict, Optional

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms

import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

# ---------------------------------------------------------
# 0. 기본 설정
# ---------------------------------------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[INFO] Using device: {device}")

IMG_SIZE = 224

preprocess = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],  # ImageNet 평균
        std=[0.229, 0.224, 0.225],
    ),
])

# ResNet50 백본 (마지막 FC 이전의 특징벡터만 사용)
base_model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
feature_extractor = nn.Sequential(*list(base_model.children())[:-1])  # GAP 직전까지
feature_extractor.eval().to(device)


# ---------------------------------------------------------
# 1. 이미지 임베딩 유틸
# ---------------------------------------------------------
def load_image(path: str) -> torch.Tensor:
    img = Image.open(path).convert("RGB")
    return preprocess(img).unsqueeze(0).to(device)


@torch.no_grad()
def embed_image(path: str) -> torch.Tensor:
    """단일 이미지 -> L2 정규화된 특징벡터 (dim=2048)."""
    img = load_image(path)
    feat = feature_extractor(img)       # (1, 2048, 1, 1)
    feat = feat.squeeze()               # (2048,)
    feat = F.normalize(feat, dim=0)
    return feat.cpu()


def cosine(a: torch.Tensor, b: torch.Tensor) -> float:
    return float(F.cosine_similarity(a.unsqueeze(0), b.unsqueeze(0)).item())


def build_proto(pattern: str) -> Optional[torch.Tensor]:
    """패턴(glob)으로 이미지 모아 평균 프로토타입 생성."""
    paths = sorted(glob.glob(pattern))
    if not paths:
        print(f"[WARN] No images found for pattern: {pattern}")
        return None
    embs = [embed_image(p) for p in paths]
    proto = torch.stack(embs, dim=0).mean(0)
    proto = F.normalize(proto, dim=0)
    print(f"[INFO] Proto built from {len(paths)} images for {pattern}")
    return proto


# ---------------------------------------------------------
# 2. 폴더 구조 정의
# ---------------------------------------------------------
DATA_ROOT = "data"  # Colab에서 이 프로젝트 폴더 기준

DIR_REFS = os.path.join(DATA_ROOT, "refs_4910E")
DIR_STOCK = os.path.join(DATA_ROOT, "stock_4910E")
DIR_CANDS = os.path.join(DATA_ROOT, "candidates_4910E")

DIR_PREM_VIEW    = os.path.join(DATA_ROOT, "premium_views_4910E")
DIR_PREM_KITCHEN = os.path.join(DATA_ROOT, "premium_kitchen_4910E")
DIR_PREM_BATH    = os.path.join(DATA_ROOT, "premium_bath_4910E")
DIR_PREM_OPTION  = os.path.join(DATA_ROOT, "premium_option_4910E")


def list_images(folder: str) -> List[str]:
    return sorted(
        glob.glob(os.path.join(folder, "*.jpg"))
        + glob.glob(os.path.join(folder, "*.jpeg"))
        + glob.glob(os.path.join(folder, "*.png"))
    )


# ---------------------------------------------------------
# 3. 프로토타입 및 후보 매물 스코어링
# ---------------------------------------------------------
def compute_scores() -> pd.DataFrame:
    """4910E 프로토타입 기반 전체 후보 매물 스코어링."""
    # 3-1) 프로토타입들 생성
    proto_main   = build_proto(os.path.join(DIR_REFS, "*.jpg"))
    if proto_main is None:
        raise RuntimeError("refs_4910E 폴더에 4910호 실사진을 넣어주세요.")

    proto_stock  = build_proto(os.path.join(DIR_STOCK, "*.jpg"))
    proto_view   = build_proto(os.path.join(DIR_PREM_VIEW, "*.jpg"))    # optional
    proto_kitch  = build_proto(os.path.join(DIR_PREM_KITCHEN, "*.jpg")) # optional
    proto_bath   = build_proto(os.path.join(DIR_PREM_BATH, "*.jpg"))    # optional
    proto_option = build_proto(os.path.join(DIR_PREM_OPTION, "*.jpg"))  # optional

    cand_paths = list_images(DIR_CANDS)
    if not cand_paths:
        raise RuntimeError("candidates_4910E 폴더에 네이버/유튜브 후보 매물 사진을 넣어주세요.")

    rows = []

    for path in cand_paths:
        emb = embed_image(path)

        sim_main = cosine(emb, proto_main)
        sim_stock = cosine(emb, proto_stock) if proto_stock is not None else float("nan")

        premium_parts = []
        if proto_view is not None:
            premium_parts.append(cosine(emb, proto_view))
        if proto_kitch is not None:
            premium_parts.append(cosine(emb, proto_kitch))
        if proto_bath is not None:
            premium_parts.append(cosine(emb, proto_bath))
        if proto_option is not None:
            premium_parts.append(cosine(emb, proto_option))

        premium_score = float(np.mean(premium_parts)) if premium_parts else float("nan")

        # 허위/가짜 의심 점수: "책자/스톡에 더 가깝고 본인 실사진과는 먼 경우"
        if proto_stock is not None:
            fake_score = max(0.0, sim_stock - sim_main)
        else:
            fake_score = float("nan")

        is_suspect = (not np.isnan(fake_score)) and (fake_score > 0.1) and (sim_main < 0.70)

        listing_id = os.path.splitext(os.path.basename(path))[0]

        rows.append(
            dict(
                listing_id=listing_id,
                image_path=path,
                sim_4910E=sim_main,
                sim_stock=sim_stock,
                premium_score=premium_score,
                fake_score=fake_score,
                is_suspect=is_suspect,
            )
        )

    df = pd.DataFrame(rows)
    df = df.sort_values("sim_4910E", ascending=False).reset_index(drop=True)
    return df


# ---------------------------------------------------------
# 4. Top 10 리포트 생성 (표 + 그래프 + PDF)
# ---------------------------------------------------------
def plot_top10_bar(df_top: pd.DataFrame):
    """Top 10 매물의 유사도 & 프리미엄 점수 막대 그래프."""
    x = np.arange(len(df_top))
    labels = df_top["listing_id"].tolist()
    sim_vals = df_top["sim_4910E"].tolist()
    prem_vals = df_top["premium_score"].tolist()

    plt.figure(figsize=(10, 5))
    width = 0.35
    plt.bar(x - width/2, sim_vals, width, label="Similarity to 4910E")
    plt.bar(x + width/2, prem_vals, width, label="Premium Score")

    plt.xticks(x, labels, rotation=45, ha="right")
    plt.ylabel("Score")
    plt.title("Top 10 – Similarity & Premium Score")
    plt.legend()
    plt.tight_layout()


def plot_premium_only(df_top: pd.DataFrame):
    """Top 10 매물의 프리미엄 점수만 별도 시각화."""
    labels = df_top["listing_id"].tolist()
    prem_vals = df_top["premium_score"].tolist()

    plt.figure(figsize=(10, 5))
    plt.bar(labels, prem_vals)
    plt.xticks(rotation=45, ha="right")
    plt.ylabel("Premium Score")
    plt.title("Top 10 – Premium Score Only")
    plt.tight_layout()


def make_pdf_report(df: pd.DataFrame, out_path: str = "report_4910E_top10.pdf"):
    """Top 10 유사 매물에 대한 PDF 리포트 생성."""
    top = df.head(10).copy()

    with PdfPages(out_path) as pdf:
        # 페이지 1: 개요 텍스트
        fig1 = plt.figure(figsize=(8.27, 11.69))  # A4 세로
        txt = (
            "Signiel Residence 4910E – Prototype Matching Report\n\n"
            "1. 4910호 E타입 실사진을 기반으로 이미지 프로토타입을 생성했습니다.\n"
            "2. 네이버 확정매물/후보 사진과의 코사인 유사도를 계산했습니다.\n"
            "3. 책자/홍보컷과의 유사도를 이용해 '스톡/허위 가능성'을 추정했습니다.\n"
            "4. 프리미엄 뷰/주방/욕실/옵션 폴더를 기반으로 프리미엄 점수를 산출했습니다.\n\n"
            "이 리포트는 4910호와 가장 유사한 상위 10개 매물의 점수를 요약한 것입니다."
        )
        fig1.text(0.05, 0.95, txt, va="top", wrap=True)
        pdf.savefig(fig1)
        plt.close(fig1)

        # 페이지 2: 표 요약
        fig2 = plt.figure(figsize=(11.69, 8.27))  # A4 가로
        plt.axis("off")
        cols = ["listing_id", "sim_4910E", "premium_score", "fake_score", "is_suspect"]
        table_data = [cols] + top[cols].round(3).values.tolist()
        table = plt.table(
            cellText=table_data,
            loc="center",
            cellLoc="center",
        )
        table.auto_set_font_size(True)
        table.scale(1.2, 1.2)
        pdf.savefig(fig2)
        plt.close(fig2)

        # 페이지 3: 유사도 + 프리미엄 점수 막대그래프
        plot_top10_bar(top)
        pdf.savefig()
        plt.close()

        # 페이지 4: 프리미엄 점수만 별도 시각화
        plot_premium_only(top)
        pdf.savefig()
        plt.close()

    print(f"[INFO] PDF report saved to: {out_path}")


# ---------------------------------------------------------
# 5. 메인 실행 진입점
# ---------------------------------------------------------
def main():
    print("[STEP] Compute scores for all candidate listings...")
    df = compute_scores()
    print("[INFO] 전체 후보 매물 수:", len(df))
    print(df.head(10))

    print("[STEP] Generate PDF report for Top 10...")
    make_pdf_report(df, out_path="report_4910E_top10.pdf")
    print("[DONE] report_4910E_top10.pdf 생성 완료")


if __name__ == "__main__":
    main()


[INFO] Using device: cpu
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 171MB/s]


[STEP] Compute scores for all candidate listings...
[WARN] No images found for pattern: data/refs_4910E/*.jpg


RuntimeError: refs_4910E 폴더에 4910호 실사진을 넣어주세요.

*  다음 추천 진행 순서

   **  위 Signiel_4910E_ALL.zip을 다운로드해서 구글 드라이브에 업로드 → 압축 해제

Colab에서 폴더 열고,

   **  data/refs_4910E / stock_4910E / candidates_4910E / premium_*_4910E 폴더들을 만들고

*사진들을 직접 분류해서 넣기

   **signiel_4910E_prototype_colab.py 열어서 맨 아래 main()까지 실행

   *** report_4910E_top10.pdf 가 생성되면 →

   ***팀 발표 / PPT(25페이지 뼈대)에 바로 반영


---



“4910호 기준 프리미엄 스코어링 시스템” 완성 🎯

   **  여기까지 세팅하시다가,

   **  폴더 분류 기준

   ***   fake/허위 기준(임계값) 조정

   ***   리포트/PPT에 넣을 문구


---


같은 부분을 더 다듬고 싶으시면, 결과 캡처만 가져와 주시면 거기에 맞춰서 발표용 문구까지 같이 정리해 드릴게요.